In [1]:
# Import packages

import kagglehub
import os
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
#import joblib
import lightgbm as lgb
#import mlflow
#import optuna


In [2]:
# Get project location
parent_dir = os.path.dirname(os.getcwd())

input_folder_dir = fr"{parent_dir}\input_files"
test_output_dir = fr"{parent_dir}\test_output"

# Create folders if they don't exist
if not os.path.exists(input_folder_dir):
    os.makedirs(input_folder_dir)

if not os.path.exists(test_output_dir):
    os.makedirs(test_output_dir)

    
# Download titanic competition dataset if not already downloaded

if not os.listdir(input_folder_dir):
    download_path = kagglehub.competition_download("spaceship-titanic",output_dir=rf"{parent_dir}\input_files")

df = {}
df['train'] = pd.read_csv(os.path.join(input_folder_dir, "train.csv"))
df['test'] = pd.read_csv(os.path.join(input_folder_dir, "test.csv"))

In [3]:
# Describe train and test sets

for x in df.keys():
    print(f"{x} size: {df[x].shape}")
    
          

train size: (8693, 14)
test size: (4277, 13)


In [4]:
for x in df.keys():
    print(f"{x} dtypes: \n {df[x].dtypes} \n")
    

train dtypes: 
 PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object 

test dtypes: 
 PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
dtype: object 



In [5]:
for x in df.keys():
    print(f"{x} summary: {df[x].describe()}")

train summary:                Age   RoomService     FoodCourt  ShoppingMall           Spa  \
count  8514.000000   8512.000000   8510.000000   8485.000000   8510.000000   
mean     28.827930    224.687617    458.077203    173.729169    311.138778   
std      14.489021    666.717663   1611.489240    604.696458   1136.705535   
min       0.000000      0.000000      0.000000      0.000000      0.000000   
25%      19.000000      0.000000      0.000000      0.000000      0.000000   
50%      27.000000      0.000000      0.000000      0.000000      0.000000   
75%      38.000000     47.000000     76.000000     27.000000     59.000000   
max      79.000000  14327.000000  29813.000000  23492.000000  22408.000000   

             VRDeck  
count   8505.000000  
mean     304.854791  
std     1145.717189  
min        0.000000  
25%        0.000000  
50%        0.000000  
75%       46.000000  
max    24133.000000  
test summary:                Age   RoomService     FoodCourt  ShoppingMall          

In [6]:
for x in df.keys():
    print(f"{x} head: \n {df[x].head()}")
    

train head: 
   PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4    

**Create a simple random forest classifier model as a baseline:**

In [7]:
# start with naive numeric-only model

X_train = df['train'].select_dtypes(exclude = ["object"]).drop(columns = ['Transported']).copy()
y_train = df['train']['Transported'].copy()
X_test = df['test'].select_dtypes(exclude = ["object"]).copy()

# use two metrics for now
metrics = ["accuracy", "roc_auc"]

# pipeline
initial_random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100, max_depth=5, random_state=1
            ),
        ),
    ]
)


In [8]:
# run 5 fold cv 
cv_results = cross_validate(
    initial_random_forest_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=metrics,
    return_train_score=True,
)

print(f"Train set - Mean CV accuracy: {np.mean(cv_results['train_accuracy']):.4f}")
print(f"Validation set - Mean CV accuracy: {np.mean(cv_results['test_accuracy']):.4f}")

print(f"Train set - Mean AUC ROC accuracy: {np.mean(cv_results['train_roc_auc']):.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {np.mean(cv_results['test_roc_auc']):.4f}")


Train set - Mean CV accuracy: 0.7989
Validation set - Mean CV accuracy: 0.7876
Train set - Mean AUC ROC accuracy: 0.8458
Validation set - Mean AUC ROC accuracy: 0.8345


In [9]:
# Initial submission

initial_random_forest_pipeline.fit(X_train, y_train)

final_preds = initial_random_forest_pipeline.predict(X_test)

submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Transported": final_preds})


print(submission['Transported'].value_counts())

    
submission.to_csv(rf"{test_output_dir}/1_init_rf_submission.csv", index=False)


Transported
True     2232
False    2045
Name: count, dtype: int64


**Fairly poor result (0.79261 - rank 1600 out of 2000) Now try a grid search to improve:**

In [10]:
param_grid = {
    "model__n_estimators": [45, 50, 55],
    "model__max_depth": [5, 6,7],
    "model__min_samples_split": [5,7,9,11],
}

grid_search = GridSearchCV(
    estimator=initial_random_forest_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring={"accuracy": "accuracy","roc_auc": "roc_auc"},
    refit="accuracy",
    return_train_score=True,
    verbose=1,
)

# run the grid search
grid_search.fit(X_train, y_train)

best_index = grid_search.best_index_
results = grid_search.cv_results_

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# 
final_preds = grid_search.predict(X_test)
submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Transported": final_preds})
print(submission['Transported'].value_counts())
submission.to_csv(rf"{test_output_dir}/2_cvgrid_rf_submission.csv", index=False)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters Found: {'model__max_depth': 7, 'model__min_samples_split': 11, 'model__n_estimators': 55}
Train set - Mean CV accuracy: 0.8129
Validation set - Mean CV accuracy: 0.7934
Train set - Mean AUC ROC accuracy: 0.8669
Validation set - Mean AUC ROC accuracy: 0.8412
Transported
True     2361
False    1916
Name: count, dtype: int64


**As with the standard Titanic comp, this yields an extremely slight improvement: 0.79752. Again, try some feature engineering, add categorical variables and imputation flags**


In [11]:
# Check for missing:

df['train'].isna().sum(axis=0)

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [12]:
# Get a better understanding of Cabin

for s in ['train','test']:
    
    cabin_parts = df[s]['Cabin'].str.split("/", expand=True)
    cabin_parts.columns = ['Deck', 'Num', 'Side']
    
# Want to make sure there are none that have partial information
    print(f"{s.title()} set missing some info: {cabin_parts.isna().any(axis=1).sum()}") 
    print(f"{s.title()} set missing all info: {cabin_parts.isna().all(axis=1).sum()}")
    
# So either all 3 parts of Cabin info appear, or none do
# Want to make it's sure fine to impute missing with U/U/U

    for c in cabin_parts.columns:
        if (cabin_parts[c] == "U").sum() > 0:
            print(f"In {s.title()} set, U appears in {c}")
        else:
            print(f"In {s.title()} set, U doesn't appear in {c}")

# Get some more info about the possible values of each
combined = pd.concat([df['train'],df['test']])
cabin_parts_combined = combined['Cabin'].str.split("/", expand=True)
cabin_parts_combined.columns = ['Deck', 'Num', 'Side']

for col in ['Deck', 'Num', 'Side']:
    print(f"{col}: \n", cabin_parts_combined[col].value_counts().sort_values(ascending=False))


    

Train set missing some info: 199
Train set missing all info: 199
In Train set, U doesn't appear in Deck
In Train set, U doesn't appear in Num
In Train set, U doesn't appear in Side
Test set missing some info: 100
Test set missing all info: 100
In Test set, U doesn't appear in Deck
In Test set, U doesn't appear in Num
In Test set, U doesn't appear in Side
Deck: 
 Deck
F    4239
G    3781
E    1323
B    1141
C    1102
D     720
A     354
T      11
Name: count, dtype: int64
Num: 
 Num
82      34
56      28
4       28
31      27
95      27
        ..
1837     1
1834     1
1832     1
1830     1
1890     1
Name: count, Length: 1894, dtype: int64
Side: 
 Side
S    6381
P    6290
Name: count, dtype: int64


In [13]:
# Reload from raw
X_train = df['train'].copy()
y_train = df['train']['Transported'].copy()
X_test = df['test'].copy()

# flag for if age is estimated in given data
X_train['est_age_flag'] = np.where(X_train['Age'].notna() & (abs(X_train['Age'] - 0.5) - np.floor(X_train['Age'])) <0.1   , 1 , 0 )
X_test['est_age_flag'] = np.where(X_test['Age'].notna() & (abs(X_test['Age'] - 0.5) - np.floor(X_test['Age'])) <0.1 , 1 , 0 )

# flag for if age is imputed
X_train['imp_age_flag'] = np.where(X_train['Age'].isna(), 1 , 0 )
X_test['imp_age_flag'] = np.where(X_test['Age'].isna(), 1 , 0 )

# manually impute age
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_test['Age'] = X_test['Age'].fillna(X_train['Age'].median())

# Split the components of Cabin into deck/num/side and inpute
X_train['deck'] = X_train['Cabin'].str.split("/", expand=True)[0].fillna("U")
X_test['deck'] = X_test['Cabin'].str.split("/", expand=True)[0].fillna("U")

X_train['side'] = X_train['Cabin'].str.split("/", expand=True)[2].fillna("U")
X_test['side'] = X_test['Cabin'].str.split("/", expand=True)[2].fillna("U")

# flag for if cabin number is imputed
X_train['imp_num'] = np.where(X_train['Cabin'].isna(),1,0)
X_test['imp_num'] = np.where(X_test['Cabin'].isna(),1,0)

# Now impute cabin number
train_num_str = X_train['Cabin'].str.split("/", expand=True)[1]
test_num_str = X_test['Cabin'].str.split("/", expand=True)[1]

num_median = train_num_str.astype(float).median()

X_train['num'] = train_num_str.fillna(num_median).astype("int64")
X_test['num'] = test_num_str.fillna(num_median).astype("int64")

# Split up PassengerId

X_train['group'] = X_train['PassengerId'].str.split("_", expand=True)[0]
X_test['group'] = X_test['PassengerId'].str.split("_", expand=True)[0]

X_train['group_num'] = X_train['PassengerId'].str.split("_", expand=True)[1]
X_test['group_num'] = X_test['PassengerId'].str.split("_", expand=True)[1]

# Impute HomePlanet
X_train['HomePlanet'] = X_train['HomePlanet'].fillna("U")
X_test['HomePlanet'] = X_test['HomePlanet'].fillna("U")

# Turn CryoSleep into Y/N string so we can impute with an unknown "U" value
X_train['CryoSleep_str'] = np.where(X_train['CryoSleep'],"Y","N")
X_test['CryoSleep_str'] = np.where(X_test['CryoSleep'],"Y","N")

X_train['CryoSleep_str'] = X_train['CryoSleep_str'].fillna("U")
X_test['CryoSleep_str'] = X_test['CryoSleep_str'].fillna("U")

# Impute VIP
X_train['VIP_str'] = np.where(X_train['VIP'],"Y","N")
X_test['VIP_str'] = np.where(X_test['VIP'],"Y","N")
X_train['VIP_str'] = X_train['VIP_str'].fillna("U")
X_test['VIP_str'] = X_test['VIP_str'].fillna("U")

# Impute Destination
X_train['Destination'] = X_train['Destination'].fillna("U")
X_test['Destination'] = X_test['Destination'].fillna("U")

# Drop unwanted columns
X_train = X_train.drop(columns = ['Name','Cabin','Transported','CryoSleep','PassengerId','VIP'])
X_test = X_test.drop(columns = ['Name','Cabin','CryoSleep','PassengerId','VIP'])

# define categorical and numeric features
numeric_features = X_train.select_dtypes(exclude=['object', 'string']).columns
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns




In [21]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="passthrough",
)
full_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor),
           ("classifier", RandomForestClassifier(random_state=1))]
)

param_distributions = {
    "classifier__n_estimators": [45, 50, 55,100,200,300,500,750],
    "classifier__max_depth": [2,4, 5, 6,7,8,12,None],
    "classifier__min_samples_split": [5,7,9, 10,11,20],
    "classifier__min_samples_leaf": [5,8,15],
    "classifier__random_state": [1],
}
 
grid_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring={"accuracy": "accuracy","roc_auc": "roc_auc"},
    refit="roc_auc",
    return_train_score=True,
    n_jobs=-1,
    random_state=1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# 
final_preds = grid_search.predict(X_test)
submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Transported": final_preds})
print(submission['Transported'].value_counts())
submission.to_csv(rf"{test_output_dir}/3_more_features_rf_submission.csv", index=False)

Best Parameters Found: {'classifier__random_state': 1, 'classifier__n_estimators': 500, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 8, 'classifier__max_depth': 12}
Train set - Mean CV accuracy: 0.8129
Validation set - Mean CV accuracy: 0.7934
Train set - Mean AUC ROC accuracy: 0.8669
Validation set - Mean AUC ROC accuracy: 0.8412
Transported
False    2281
True     1996
Name: count, dtype: int64


In [22]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="passthrough",
)
full_pipeline_lgb = Pipeline(
    steps=[("preprocessor", preprocessor),
           ("classifier", lgb.LGBMClassifier(verbosity=-1))]
)

param_distributions = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__num_leaves": [15, 31, 63],            # Opened up (Default is 31)
    "classifier__max_depth": [3, 5, 7, -1],             # Allow deeper trees (-1 means unlimited depth)
    "classifier__min_child_samples": [10, 20, 30, 50],  # Adjusted down so small trees can still form split points
    "classifier__random_state": [1],
}

grid_search_lgb = RandomizedSearchCV(
    estimator=full_pipeline_lgb,
    param_distributions=param_distributions, # FIXED from param_distributions
    n_iter=200,
    cv=5,
    scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
    refit="accuracy",
    return_train_score=True,
    n_jobs=-1,
    random_state=1
)

grid_search_lgb.fit(X_train, y_train)


lgb_results = grid_search_lgb.cv_results_
lgb_best_index = grid_search_lgb.best_index_

print(f"Best Parameters Found: {grid_search_lgb.best_params_}\n")
print(f"Train set - Mean CV accuracy: {lgb_results['mean_train_accuracy'][lgb_best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {lgb_results['mean_test_accuracy'][lgb_best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {lgb_results['mean_train_roc_auc'][lgb_best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {lgb_results['mean_test_roc_auc'][lgb_best_index]:.4f}")


final_preds = grid_search_lgb.predict(X_test)
submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Transported": final_preds})
print(submission['Transported'].value_counts())
submission.to_csv(rf"{test_output_dir}/4_lightgbm_submission.csv", index=False)

Best Parameters Found: {'classifier__random_state': 1, 'classifier__num_leaves': 63, 'classifier__n_estimators': 100, 'classifier__min_child_samples': 50, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.1}

Train set - Mean CV accuracy: 0.8204
Validation set - Mean CV accuracy: 0.7966
Train set - Mean AUC ROC accuracy: 0.9130
Validation set - Mean AUC ROC accuracy: 0.8879
Transported
True     2308
False    1969
Name: count, dtype: int64


D:\Python\venv\main_venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
